In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [39]:
!pwd

/content


In [40]:
a=28*64
a

1792

In [41]:
# help(datasets.mnist.MNIST)
# help(DataLoader)
# help(nn.CrossEntropyLoss())

In [42]:
# ----------------------
# 1. 超参数设置
# ----------------------
batch_size=64   
input_size=784
hidden_size1=256
hidden_size2=128
output_size=10
lr=0.01
num_epochs=5



In [43]:
# ----------------------
# 2. 数据加载与预处理
# ----------------------
# 定义预处理：转为Tensor + 归一化
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)),
]
)

train_dataset=datasets.mnist.MNIST(root='./data',train=True,download=True,transform=transform)
test_dataset=datasets.mnist.MNIST(root='./data',train=False,download=True,transform=transform)

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)


In [44]:
x,y=train_dataset[0]
x.shape,y

(torch.Size([1, 28, 28]), 5)

In [45]:
# images_batch, labels_batch = next(iter(train_loader))
# print(f"\n一个batch的图像形状: {images_batch.shape}")  # 输出: torch.Size([64, 1, 28, 28])
# print(f"一个batch的标签形状: {labels_batch.shape}")    # 输出: torch.Size([64])

In [46]:
# ----------------------
# 3. 定义模型
# ----------------------
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet,self).__init__()
        self.layers=nn.Sequential(
            nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5,padding=2),
            nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2,stride=2),
        )

    def forward(self,X):
        return self.layers(X)


class LeNet(nn.Module):
    def __init__(self):
        super(LeNet,self).__init__()
        self.layers=nn.Sequential(
            # 28@28
            nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5,padding=2),
            # 6@28@28
            nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2,stride=2),
            # 6@14@14
            nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5),
            # 16@10@10
            nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2,stride=2),
            # 16@5@5
            # nn.Flatten(),
            nn.Linear(16*5*5,120),
            nn.ReLU(),
            nn.Linear(120,84),
            nn.ReLU(),
            nn.Linear(84,10),
        )
        
    def forward(self,input):
        output=self.layers(input)
        return output
        
        
model=LeNet()
        

In [47]:
# ----------------------
# 4. 定义损失函数和优化器
# ----------------------
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr)


In [48]:
# ----------------------
# 5. 训练模型
# ----------------------
for epoch in range(num_epochs):
    for i,(features,labels) in enumerate(train_loader):
        # features=features.reshape(-1,28,28)
        
        output=model(features)
        loss=criterion(output,labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if((i+1)%100==0):
            print(f'epoch:{epoch+1}/{num_epochs}, step:{i+1}/{len(train_loader)}, loss:{loss}')            
        
        
        


RuntimeError: mat1 and mat2 shapes cannot be multiplied (5120x5 and 400x120)

In [ ]:
# ----------------------
# 6. 测试模型
# ----------------------


In [ ]:


# # ----------------------
# # 7. 可视化结果
# # ----------------------
# # 绘制损失曲线
# plt.figure(figsize=(10, 4))
# plt.subplot(1, 2, 1)
# plt.plot(loss_list)
# plt.title('训练损失')
# plt.xlabel('步骤 (x100)')
# plt.ylabel('Loss')

# # 可视化一些预测结果
# plt.subplot(1, 2, 2)
# examples = iter(test_loader)
# images, labels = next(examples)
# images = images.reshape(-1, 28, 28)  # 恢复为28x28图像

# # 预测前6张图
# outputs = model(images[:6].reshape(-1, 784))
# _, preds = torch.max(outputs, 1)

# for i in range(6):
#     plt.subplot(2, 3, i+1)
#     plt.imshow(images[i], cmap='gray')
#     plt.title(f'真实: {labels[i].item()}\n预测: {preds[i].item()}')
#     plt.axis('off')

# plt.tight_layout()
# plt.show()